In [ ]:
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from PIL import Image
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset, random_split
from torch.utils.tensorboard import SummaryWriter

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        # print(os.path.join(dirname, filename))
        pass

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATA_PATH = "/kaggle/input/face-fingerprint-dataset"
IMAGE_SIZE = 160
BATCH_SIZE = 32
EPOCHS = 10
EMBED_DIM = 512
LR = 1e-4

In [ ]:
transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ]
)

In [ ]:
class MultiModalDataset(Dataset):

    def __init__(self, root_dir, transform=None):
        self.face_dir = os.path.join(root_dir, "face")
        self.finger_dir = os.path.join(root_dir, "fingerprint")
        self.files = sorted(os.listdir(self.face_dir))
        self.label_map = {fname: idx for idx, fname in enumerate(self.files)}
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx]
        face = Image.open(os.path.join(self.face_dir, fname)).convert("RGB")
        finger = Image.open(os.path.join(self.finger_dir, fname)).convert("RGB")

        if self.transform:
            face = self.transform(face)
            finger = self.transform(finger)

        label = self.label_map[fname]
        return face, finger, label

In [ ]:
dataset = MultiModalDataset(DATA_PATH, transform)
num_classes = len(dataset)

In [ ]:
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

In [ ]:
train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size])

In [ ]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

In [ ]:
class BahdanauAttention(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.W1 = nn.Linear(dim, dim)
        self.W2 = nn.Linear(dim, dim)
        self.V = nn.Linear(dim, 1)

    def forward(self, query, values):
        score = self.V(torch.tanh(self.W1(query) + self.W2(values)))
        weights = torch.softmax(score, dim=1)
        context = torch.sum(weights * values, dim=1)
        return context

In [ ]:
class GettingUnit(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Linear(dim, dim * 2)
        self.glu = nn.GLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc(x)
        x = self.glu(x)
        return self.sigmoid(x)

In [ ]:
class ModalityEncoder(nn.Module):

    def __init__(self, embed_dim):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        self.backbone.classifier = nn.Identity()

        for p in self.backbone.parameters():
            p.requires_grad = False

        self.projection = nn.Linear(1280, embed_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.attn = BahdanauAttention(embed_dim)

        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, embed_dim),
        )

        self.getting = GettingUnit(embed_dim)

    def forward(self, x):
        B = x.size(0)
        feat = self.backbone(x)
        feat = self.projection(feat)
        feat = feat.unsqueeze(1)

        cls = self.cls_token.expand(B, -1, -1)
        context = self.attn(cls, feat)

        out = self.mlp(context)
        out = self.getting(out)

        return F.normalize(out, dim=1)

In [ ]:
class FusionModule(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.attn = BahdanauAttention(dim)
        self.getting = GettingUnit(dim)

    def forward(self, face, finger):
        combined = torch.stack([face, finger], dim=1)
        query = combined[:, 0:1, :]
        context = self.attn(query, combined)
        fused = self.getting(context)
        return F.normalize(fused, dim=1)

In [ ]:
class MultiModalModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.face_enc = ModalityEncoder(EMBED_DIM)
        self.finger_enc = ModalityEncoder(EMBED_DIM)
        self.fusion = FusionModule(EMBED_DIM)

    def forward(self, face, finger):
        f1 = self.face_enc(face)
        f2 = self.finger_enc(finger)
        return self.fusion(f1, f2)

In [ ]:
# Get one sample
face, finger, label = dataset[0]

# Convert tensor → numpy image
face_img = TF.to_pil_image(face)
finger_img = TF.to_pil_image(finger)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(face_img)
plt.title(f"Face\nLabel: {label}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(finger_img)
plt.title(f"Fingerprint\nLabel: {label}")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
idx = random.randint(0, len(dataset) - 1)
face, finger, label = dataset[idx]

face_img = TF.to_pil_image(face)
finger_img = TF.to_pil_image(finger)

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(face_img)
plt.title(f"Face\nLabel: {label}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(finger_img)
plt.title(f"Fingerprint\nLabel: {label}")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def show_samples(dataset, num_samples=4):
    plt.figure(figsize=(8, 4 * num_samples))

    for i in range(num_samples):
        idx = random.randint(0, len(dataset) - 1)
        face, finger, label = dataset[idx]

        face_img = TF.to_pil_image(face)
        finger_img = TF.to_pil_image(finger)

        plt.subplot(num_samples, 2, 2 * i + 1)
        plt.imshow(face_img)
        plt.title(f"Face - Label {label}")
        plt.axis("off")

        plt.subplot(num_samples, 2, 2 * i + 2)
        plt.imshow(finger_img)
        plt.title(f"Fingerprint - Label {label}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()


show_samples(dataset, num_samples=5)

In [ ]:
model = MultiModalModel().to(device)
classifier = nn.Linear(EMBED_DIM, num_classes).to(device)

In [ ]:
optimizer = torch.optim.Adam(
    list(model.parameters()) + list(classifier.parameters()), lr=LR
)

In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
writer = SummaryWriter("runs/multimodal_experiment")

In [ ]:
train_losses, val_losses = [], []
train_accs, val_accs = [], []

In [ ]:
def evaluate(loader):
    model.eval()
    classifier.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for face, finger, labels in loader:
            face, finger, labels = face.to(device), finger.to(device), labels.to(device)
            emb = model(face, finger)
            out = classifier(emb)
            loss = criterion(out, labels)

            total_loss += loss.item()
            preds = torch.argmax(out, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            probs = torch.softmax(out, dim=1)
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    avg_loss = total_loss / len(loader)
    acc = correct / total

    return avg_loss, acc, torch.cat(all_probs), torch.cat(all_labels)

In [ ]:
for epoch in range(EPOCHS):
    model.train()
    classifier.train()
    total_loss = 0
    correct = 0
    total = 0

    for face, finger, labels in train_loader:
        face, finger, labels = face.to(device), finger.to(device), labels.to(device)

        optimizer.zero_grad()
        emb = model(face, finger)
        out = classifier(emb)
        loss = criterion(out, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(out, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = total_loss / len(train_loader)
    train_acc = correct / total

    val_loss, val_acc, _, _ = evaluate(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    writer.add_scalar("Loss/Train", train_loss, epoch)
    writer.add_scalar("Loss/Val", val_loss, epoch)
    writer.add_scalar("Accuracy/Train", train_acc, epoch)
    writer.add_scalar("Accuracy/Val", val_acc, epoch)

    print(f"Epoch {epoch+1}\n Train Loss: {train_loss:.4f}\tValidation Loss: {val_loss:.4f}")

In [ ]:
test_loss, test_acc, probs, labels = evaluate(test_loader)
print("Test Accuracy:", test_acc)

In [ ]:
labels_onehot = F.one_hot(labels, num_classes=num_classes).numpy()
roc_auc = roc_auc_score(labels_onehot, probs.numpy(), multi_class="ovr")
print("ROC-AUC:", roc_auc)

In [ ]:
plt.figure()
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.title("Loss vs Epoch")
plt.show()

In [ ]:
plt.figure()
plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Val Accuracy")
plt.legend()
plt.title("Accuracy vs Epoch")
plt.show()

In [ ]:
print("\nRun this in terminal:")
print("tensorboard --logdir=runs")

In [ ]:
from tensorboard.backend.event_processing import event_accumulator
import matplotlib.pyplot as plt
import os

log_dir = "runs/multimodal_experiment"

# Load event file
event_files = [os.path.join(log_dir, f) for f in os.listdir(log_dir)]
event_file = event_files[0]

ea = event_accumulator.EventAccumulator(event_file)
ea.Reload()

# Get all scalar tags
print("Available scalar tags:")
print(ea.Tags()["scalars"])

In [ ]:
def get_scalar(tag):
    events = ea.Scalars(tag)
    steps = [e.step for e in events]
    values = [e.value for e in events]
    return steps, values

In [ ]:
train_loss_steps, train_loss = get_scalar("Loss/Train")
val_loss_steps, val_loss = get_scalar("Loss/Val")
train_acc_steps, train_acc = get_scalar("Accuracy/Train")
val_acc_steps, val_acc = get_scalar("Accuracy/Val")

In [ ]:
train_error = [1 - a for a in train_acc]
val_error = [1 - a for a in val_acc]

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(train_loss_steps, train_loss, label="Train Loss")
plt.plot(val_loss_steps, val_loss, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epoch")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(train_acc_steps, train_acc, label="Train Accuracy")
plt.plot(val_acc_steps, val_acc, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Epoch")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(train_acc_steps, train_error, label="Training Error")
plt.plot(val_acc_steps, val_error, label="Validation Error")
plt.xlabel("Epoch")
plt.ylabel("Error")
plt.title("Error vs Epoch")
plt.legend()
plt.grid()
plt.show()

In [ ]:
def extract_embeddings(loader):
    model.eval()
    classifier.eval()

    embeddings = []
    labels = []

    with torch.no_grad():
        for face, finger, lbl in loader:
            face = face.to(device)
            finger = finger.to(device)

            emb = model(face, finger)
            embeddings.append(emb.cpu())
            labels.append(lbl)

    embeddings = torch.cat(embeddings)
    labels = torch.cat(labels)
    return embeddings, labels

In [ ]:
def compute_similarity_matrix(embeddings):
    embeddings = F.normalize(embeddings, dim=1)
    sim_matrix = torch.matmul(embeddings, embeddings.T)
    return sim_matrix

In [ ]:
def rank_n_accuracy(sim_matrix, labels, N=1):
    correct = 0
    num_samples = len(labels)

    for i in range(num_samples):
        similarities = sim_matrix[i]
        ranked_indices = torch.argsort(similarities, descending=True)

        # Remove self-match
        ranked_indices = ranked_indices[ranked_indices != i]

        top_n = labels[ranked_indices[:N]]

        if labels[i] in top_n:
            correct += 1

    return correct / num_samples

In [ ]:
def compute_cmc(sim_matrix, labels):
    num_samples = len(labels)
    cmc = torch.zeros(num_samples)

    for i in range(num_samples):
        similarities = sim_matrix[i]
        ranked_indices = torch.argsort(similarities, descending=True)

        ranked_indices = ranked_indices[ranked_indices != i]
        ranked_labels = labels[ranked_indices]

        correct_index = (ranked_labels == labels[i]).nonzero(as_tuple=True)[0]

        if len(correct_index) > 0:
            rank = correct_index[0]
            cmc[rank:] += 1

    cmc = cmc / num_samples
    return cmc

In [ ]:
embeddings, labels = extract_embeddings(test_loader)
sim_matrix = compute_similarity_matrix(embeddings)

rank1 = rank_n_accuracy(sim_matrix, labels, N=1)
rank5 = rank_n_accuracy(sim_matrix, labels, N=5)
rank10 = rank_n_accuracy(sim_matrix, labels, N=10)

print("Rank-1 Accuracy:", rank1)
print("Rank-5 Accuracy:", rank5)
print("Rank-10 Accuracy:", rank10)

cmc = compute_cmc(sim_matrix, labels)

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(cmc.numpy())
plt.xlabel("Rank")
plt.ylabel("Identification Rate")
plt.title("CMC Curve")
plt.grid()
plt.show()

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir /kaggle/working/runs/multimodal_experiment